<a href="https://colab.research.google.com/github/unatisaini/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/unatisaini/flyrank-internship-mi/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
X = a[FEATURES]
print("feature matrix (pages x features):", X.shape)
X.dtypes.to_frame("dtype")

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing | Type | Known when? |
|---|---|---|---|---|
| `avg_position_30d` | mean Google position over the 30 days | none after filtering | number | end of the window |
| `log_impr` | log(1 + impressions) | none | number | end of the window |
| `engagement_rate_30d` | engaged sessions ÷ sessions × 100 | never blank; 0 when there are no sessions | number | end of the window, after visits |
| `scroll_rate_30d` | scroll events ÷ pageviews × 100 (can exceed 100) | same | number | same |
| `ai_traffic_pct_30d` | AI-referred sessions ÷ sessions × 100 | same | number | same |
| `content_type` | article type | none | category | at publish |
| `main_intent` | main search intent | none | category | at publish |
| `age_tier` | page age bucket | none | category | any time |
| `freshness_tier` | time since last update | none | category | any time |
| `word_count_tier`, `char_count_tier` | length buckets | blank for some pages (5,680 rows in the full cut) | category | at publish |

The model handles blank categories natively, so nothing is imputed.

In [ ]:
notes = pd.DataFrame({
    "dtype": X.dtypes.astype(str),
    "missing": X.isna().sum(),
    "n_unique": X.nunique(),
})
notes

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
label_sources = {"ctr_30d", "clicks_30d", "needs_ctr_fix", "health_score", "is_quick_win",
                 "needs_engagement_fix", "needs_indexing", "ai_opportunity"}
print("label sources inside FEATURES:", sorted(label_sources & set(FEATURES)) or "none")

num = ["avg_position_30d", "log_impr"] + ENG + VOL
corr = a[num + ["clicks_30d"]].corr(method="spearman")["clicks_30d"].drop("clicks_30d")
print("\nSpearman correlation with clicks (visible pages):")
print(corr.sort_values(ascending=False).round(2).to_string())

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded | Why |
|---|---|
| `ctr_30d`, `clicks_30d` | they build the label |
| `needs_ctr_fix`, `health_score`, `is_quick_win`, `needs_engagement_fix`, `needs_indexing`, `ai_opportunity` | FlyRank product rules; `needs_ctr_fix` is kept only as a baseline |
| `pageviews_30d`, `sessions_30d`, `users_30d`, `engaged_sessions_30d`, `ai_sessions_30d`, `scroll_events_30d` | visit counts that start as clicks; used only in a sensitivity check |
| `impression_tier`, `position_tier` | tiers of columns already used |
| `client_has_gsc`, `client_has_ga4` | true for every row |
| `client_hash_id`, `content_hash_id` | IDs; the client ID is used only to group the split |

Privacy: every ID is hashed, and nothing in this notebook prints raw rows, URLs, domains or queries.

In [ ]:
excluded = ["ctr_30d", "clicks_30d", "needs_ctr_fix", "health_score", "is_quick_win",
            "needs_engagement_fix", "needs_indexing", "ai_opportunity", "impression_tier",
            "position_tier", "client_has_gsc", "client_has_ga4", "client_hash_id", "content_hash_id"] + VOL
print("excluded columns that slipped into FEATURES:", sorted(set(excluded) & set(FEATURES)) or "none")
print("features + excluded cover every column:",
      set(df.columns) <= set(excluded) | set(FEATURES) | {"impressions_30d"})

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.